# 问答系统 (PyTorch)

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [ ]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl
!apt install git-lfs

You will need to setup git, adapt your email and name in the following cell.

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"

You will also need to be logged in to the Hugging Face Hub. Execute the following and enter your credentials.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [1]:
from datasets import load_dataset

raw_datasets = load_dataset("squad")

In [2]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})

In [3]:
print("Context: ", raw_datasets["train"][0]["context"])
print("Question: ", raw_datasets["train"][0]["question"])
print("Answer: ", raw_datasets["train"][0]["answers"])

Context:  Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
Question:  To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?
Answer:  {'text': ['Saint Bernadette Soubirous'], 'answer_start': [515]}


In [4]:
raw_datasets["train"].filter(lambda x: len(x["answers"]["text"]) != 1)

Filter:   0%|          | 0/87599 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 0
})

In [5]:
print(raw_datasets["validation"][0]["answers"])
print(raw_datasets["validation"][2]["answers"])

{'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'answer_start': [177, 177, 177]}
{'text': ['Santa Clara, California', "Levi's Stadium", "Levi's Stadium in the San Francisco Bay Area at Santa Clara, California."], 'answer_start': [403, 355, 355]}


In [6]:
print(raw_datasets["validation"][2]["context"])
print(raw_datasets["validation"][2]["question"])

Super Bowl 50 was an American football game to determine the champion of the National Football League (NFL) for the 2015 season. The American Football Conference (AFC) champion Denver Broncos defeated the National Football Conference (NFC) champion Carolina Panthers 24–10 to earn their third Super Bowl title. The game was played on February 7, 2016, at Levi's Stadium in the San Francisco Bay Area at Santa Clara, California. As this was the 50th Super Bowl, the league emphasized the "golden anniversary" with various gold-themed initiatives, as well as temporarily suspending the tradition of naming each Super Bowl game with Roman numerals (under which the game would have been known as "Super Bowl L"), so that the logo could prominently feature the Arabic numerals 50.
Where did Super Bowl 50 take place?


In [7]:
from transformers import AutoTokenizer

model_checkpoint = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [8]:
tokenizer.is_fast

True

In [9]:
context = raw_datasets["train"][0]["context"]
question = raw_datasets["train"][0]["question"]

inputs = tokenizer(question, context)
tokenizer.decode(inputs["input_ids"])

'[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building \' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive ( and in a direct line that connects through 3 statues and the Gold Dome ), is a simple, modern stone statue of Mary. [SEP]'

In [10]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basi [SEP]
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin [SEP]
[CLS] To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France? [SEP] Next to the Main Building is the

In [11]:
inputs = tokenizer(
    question,
    context,
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)
inputs.keys()

KeysView({'input_ids': [[101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 22182, 1193, 117, 1103, 1278, 1144, 170, 2336, 1959, 119, 1335, 4184, 1103, 4304, 4334, 112, 188, 2284, 10945, 1110, 170, 5404, 5921, 1104, 1103, 6567, 2090, 119, 13301, 1107, 1524, 1104, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 1103, 171, 17506, 102], [101, 1706, 2292, 1225, 1103, 6567, 2090, 9273, 2845, 1107, 8109, 1107, 10111, 20500, 1699, 136, 102, 1103, 4304, 4334, 1105, 4749, 1122, 117, 1110, 170, 7335, 5921, 1104, 4028, 1114, 1739, 1146, 14089, 5591, 1114, 1103, 7051, 107, 159, 21462, 1566, 24930, 2508, 152, 1306, 3965, 107, 119, 5893, 1106, 1103, 4304, 4334, 1110, 1103, 19349, 1104, 1103, 11373, 4641, 119, 13301, 1481, 

In [18]:
inputs["overflow_to_sample_mapping"]

[0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3]

In [23]:
inputs = tokenizer(
    raw_datasets["train"][2:6]["question"],
    raw_datasets["train"][2:6]["context"],
    max_length=100,
    truncation="only_second",
    stride=50,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

print(f"The 4 examples gave {len(inputs['input_ids'])} features.")
print(f"Here is where each comes from: {inputs['overflow_to_sample_mapping']}.")
print(f"inputs['offset_mapping']: {inputs['offset_mapping']}")

The 4 examples gave 19 features.
Here is where each comes from: [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3].
inputs['offset_mapping']: [[(0, 0), (0, 3), (4, 12), (13, 15), (16, 19), (20, 26), (27, 32), (33, 35), (36, 41), (42, 46), (47, 49), (50, 56), (57, 59), (60, 65), (66, 75), (75, 76), (0, 0), (0, 13), (13, 15), (15, 16), (17, 20), (21, 27), (28, 31), (32, 33), (34, 42), (43, 52), (52, 53), (54, 56), (56, 58), (59, 62), (63, 67), (68, 76), (76, 77), (77, 78), (79, 83), (84, 88), (89, 91), (92, 93), (94, 100), (101, 107), (108, 110), (111, 114), (115, 121), (122, 126), (126, 127), (128, 139), (140, 142), (143, 148), (149, 151), (152, 155), (156, 160), (161, 169), (170, 173), (174, 180), (181, 183), (183, 184), (185, 187), (188, 189), (190, 196), (197, 203), (204, 206), (207, 213), (214, 218), (219, 223), (224, 226), (226, 229), (229, 232), (233, 237), (238, 241), (242, 248), (249, 250), (250, 251), (251, 254), (254, 256), (257, 259), (260, 262), (263, 264), (264, 265)

In [25]:
# 取训练集中第 2~5 条样本的答案（与前面 inputs 的 overflow_to_sample_mapping 对应）
answers = raw_datasets["train"][2:6]["answers"]

start_positions = []  # 每个特征（分块）的答案起始 token 位置
end_positions = []    # 每个特征（分块）的答案结束 token 位置

# inputs["offset_mapping"] 是每个 token 对应原始字符串的 [start, end) 字符偏移
for i, offset in enumerate(inputs["offset_mapping"]):
    # overflow_to_sample_mapping 记录当前特征属于哪个原始样本（长文本被切成多个特征）
    sample_idx = inputs["overflow_to_sample_mapping"][i]
    answer = answers[sample_idx]

    # 答案在原始 context 中的字符级起止位置
    start_char = answer["answer_start"][0]
    end_char = answer["answer_start"][0] + len(answer["text"][0])

    # sequence_ids: 0=question tokens, 1=context tokens, None=special tokens
    sequence_ids = inputs.sequence_ids(i)

    # ── 找到 context 对应的 token 范围 ──────────────────────────────────────
    idx = 0
    while sequence_ids[idx] != 1:   # 跳过 question 及开头的 special token
        idx += 1
    context_start = idx             # context 第一个 token 的位置

    while sequence_ids[idx] == 1:   # 遍历所有 context token
        idx += 1
    context_end = idx - 1           # context 最后一个 token 的位置

    # ── 判断答案是否完整落在当前分块的 context 内 ───────────────────────────
    # offset[context_start][0]: context 第一个 token 对应的起始字符
    # offset[context_end][1]:   context 最后一个 token 对应的结束字符
    if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
        # 答案超出当前分块范围，无法标注，标记为 (0, 0) 表示无答案
        start_positions.append(0)
        end_positions.append(0)
    else:
        # ── 答案在当前分块内：定位答案的起始 token ──────────────────────────
        # 从 context_start 向右走，直到 token 的起始字符 > start_char
        # 此时 idx-1 就是包含 start_char 的 token
        idx = context_start
        while idx <= context_end and offset[idx][0] <= start_char:
            idx += 1
        start_positions.append(idx - 1)

        # ── 定位答案的结束 token ─────────────────────────────────────────────
        # 从 context_end 向左走，直到 token 的结束字符 < end_char
        # 此时 idx+1 就是包含 end_char 的 token
        idx = context_end
        while idx >= context_start and offset[idx][1] >= end_char:
            idx -= 1
        end_positions.append(idx + 1)

# ── 打印结果并直观展示每个特征的标注 ────────────────────────────────────────
print(f"共 {len(start_positions)} 个特征（原始样本经分块后产生）")
print(f"start_positions: {start_positions}")
print(f"end_positions:   {end_positions}")
print()
for i, (s, e) in enumerate(zip(start_positions, end_positions)):
    sample_idx = inputs["overflow_to_sample_mapping"][i]
    answer_text = answers[sample_idx]["text"][0]
    if s == 0 and e == 0:
        print(f"  特征 {i} (样本 {sample_idx}): 答案不在此分块内 → (0, 0)")
    else:
        # 用 offset_mapping 还原 token 范围对应的字符，验证标注正确性
        char_start = inputs["offset_mapping"][i][s][0]
        char_end   = inputs["offset_mapping"][i][e][1]
        context = raw_datasets["train"][2 + sample_idx]["context"]
        decoded = context[char_start:char_end]
        print(f"  特征 {i} (样本 {sample_idx}): token [{s}, {e}] → \"{decoded}\"  (真实答案: \"{answer_text}\")")

start_positions, end_positions

共 19 个特征（原始样本经分块后产生）
start_positions: [83, 51, 19, 0, 0, 64, 27, 0, 34, 0, 0, 0, 67, 34, 0, 0, 0, 0, 0]
end_positions:   [85, 53, 21, 0, 0, 70, 33, 0, 40, 0, 0, 0, 68, 35, 0, 0, 0, 0, 0]

  特征 0 (样本 0): token [83, 85] → "the Main Building"  (真实答案: "the Main Building")
  特征 1 (样本 0): token [51, 53] → "the Main Building"  (真实答案: "the Main Building")
  特征 2 (样本 0): token [19, 21] → "the Main Building"  (真实答案: "the Main Building")
  特征 3 (样本 0): 答案不在此分块内 → (0, 0)
  特征 4 (样本 1): 答案不在此分块内 → (0, 0)
  特征 5 (样本 1): token [64, 70] → "a Marian place of prayer and reflection"  (真实答案: "a Marian place of prayer and reflection")
  特征 6 (样本 1): token [27, 33] → "a Marian place of prayer and reflection"  (真实答案: "a Marian place of prayer and reflection")
  特征 7 (样本 1): 答案不在此分块内 → (0, 0)
  特征 8 (样本 2): token [34, 40] → "a golden statue of the Virgin Mary"  (真实答案: "a golden statue of the Virgin Mary")
  特征 9 (样本 2): 答案不在此分块内 → (0, 0)
  特征 10 (样本 2): 答案不在此分块内 → (0, 0)
  特征 11 (样本 2): 答案不在此分块内 → (0, 0)
  特征

([83, 51, 19, 0, 0, 64, 27, 0, 34, 0, 0, 0, 67, 34, 0, 0, 0, 0, 0],
 [85, 53, 21, 0, 0, 70, 33, 0, 40, 0, 0, 0, 68, 35, 0, 0, 0, 0, 0])

In [26]:
idx = 0
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

start = start_positions[idx]
end = end_positions[idx]
labeled_answer = tokenizer.decode(inputs["input_ids"][idx][start : end + 1])

print(f"Theoretical answer: {answer}, labels give: {labeled_answer}")

Theoretical answer: the Main Building, labels give: the Main Building


In [ ]:
# 答案不在当前context片段的情况
idx = 4
sample_idx = inputs["overflow_to_sample_mapping"][idx]
answer = answers[sample_idx]["text"][0]

decoded_example = tokenizer.decode(inputs["input_ids"][idx])
print(f"Theoretical answer: {answer}, decoded example: {decoded_example}")

Theoretical answer: a Marian place of prayer and reflection, decoded example: [CLS] What is the Grotto at Notre Dame? [SEP] Architecturally, the school has a Catholic character. Atop the Main Building ' s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend " Venite Ad Me Omnes ". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grot [SEP]


In [28]:
max_length = 384
stride = 128


def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # Find the start and end of the context
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # If the answer is not fully inside the context, label is (0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # Otherwise it's the start and end token positions
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [29]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
len(raw_datasets["train"]), len(train_dataset)

Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

(87599, 88729)

In [30]:
def preprocess_validation_examples(examples):
    # 去除问题首尾空白
    questions = [q.strip() for q in examples["question"]]

    # ── 分词 ────────────────────────────────────────────────────────────────
    # 与训练集处理相同：对过长 context 使用滑动窗口切分成多个特征
    # return_offsets_mapping=True: 保留每个 token 对应的字符偏移，推理时用于还原答案文本
    # return_overflowing_tokens=True: 启用滑动窗口，长文本产生多个特征
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",   # 只截断 context，不截断 question
        stride=stride,              # 相邻窗口重叠 stride 个 token，避免答案被切断
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # sample_map[i] = 特征 i 来自哪个原始样本的索引（一个样本可产生多个特征）
    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []   # 记录每个特征对应的原始样本 id（字符串，用于推理时聚合答案）

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]

        # 保存原始样本的字符串 id（如 "56be4db0acb8001400a502ec"）
        # 验证时需要用它把同一问题的多个特征预测结果聚合到一起
        example_ids.append(examples["id"][sample_idx])

        # sequence_ids: 0=question, 1=context, None=special token
        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]

        # ── 关键处理：将非 context 位置的 offset 置为 None ──────────────────
        # 训练集不需要此步骤（训练时不用 offset 还原文本）
        # 验证/推理时，模型输出 start/end logits 后需要用 offset_mapping 把
        # token 位置映射回字符位置，进而从原始 context 中截取答案文本。
        # 将 question token 和 special token 的 offset 置为 None，
        # 可以在后处理中方便地过滤掉这些位置，只在 context token 中搜索答案。
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(offset)
        ]

    # 将 example_id 字段附加到 inputs 中，供后处理函数使用
    inputs["example_id"] = example_ids
    return inputs


# ── 调试：打印处理前后的对比，帮助理解 offset_mapping 的变化 ─────────────────
_demo = raw_datasets["validation"].select(range(2))
_out  = _demo.map(preprocess_validation_examples, batched=True,
                  remove_columns=_demo.column_names)

print(f"原始样本数: 2  →  产生特征数: {len(_out)}")
print()

feat = _out[0]
non_null = [(k, o) for k, o in enumerate(feat["offset_mapping"]) if o is not None]
null_cnt  = feat["offset_mapping"].count(None)
print(f"特征 0  example_id: {feat['example_id']}")
print(f"  offset_mapping 总长度  : {len(feat['offset_mapping'])}")
print(f"  None（question/special）: {null_cnt} 个  ← 推理时跳过这些位置")
print(f"  非 None（context token）: {len(non_null)} 个  ← 只在这里搜索答案")
print(f"  前 5 个非 None offset   : {non_null[:5]}")

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

原始样本数: 2  →  产生特征数: 2

特征 0  example_id: 56be4db0acb8001400a502ec
  offset_mapping 总长度  : 384
  None（question/special）: 226 个  ← 推理时跳过这些位置
  非 None（context token）: 158 个  ← 只在这里搜索答案
  前 5 个非 None offset   : [(13, [0, 5]), (14, [6, 10]), (15, [11, 13]), (16, [14, 17]), (17, [18, 20])]


In [31]:
validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)
len(raw_datasets["validation"]), len(validation_dataset)

Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

(10570, 10822)

In [32]:
small_eval_set = raw_datasets["validation"].select(range(100))
trained_checkpoint = "distilbert-base-cased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(trained_checkpoint)
eval_set = small_eval_set.map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [36]:
import torch
from transformers import AutoModelForQuestionAnswering

eval_set_for_model = eval_set.remove_columns(["example_id", "offset_mapping"])
eval_set_for_model.set_format("torch")

In [38]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# 新版 datasets 的列索引返回 Column 对象，不能直接传给 torch.tensor()
# 用 DataLoader + default_data_collator 让 HuggingFace 自动完成类型转换
dataloader = DataLoader(
    eval_set_for_model,
    batch_size=len(eval_set_for_model),
    collate_fn=default_data_collator,
)
batch = next(iter(dataloader))
batch = {k: v.to(device) for k, v in batch.items()}

trained_model = AutoModelForQuestionAnswering.from_pretrained(trained_checkpoint).to(
    device
)

with torch.no_grad():
    outputs = trained_model(**batch)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [39]:
outputs

QuestionAnsweringModelOutput(loss=None, start_logits=tensor([[ -2.2607,  -5.1783,  -5.2709,  ...,  -9.5243,  -9.5183,  -9.5288],
        [ -2.5961,  -5.5482,  -5.5313,  ...,  -9.9597,  -9.9533,  -9.9860],
        [ -3.7127,  -7.1848,  -8.5388,  ..., -11.6557, -11.6571, -11.6505],
        ...,
        [ -2.0260,  -4.4167,  -4.4980,  ...,  -8.1479,  -8.1530,  -8.1760],
        [ -4.1553,  -5.8304,  -7.1643,  ..., -10.5255, -10.5251, -10.4890],
        [ -3.2000,  -5.8162,  -6.7249,  ...,  -9.4935,  -9.5038,  -9.4871]]), end_logits=tensor([[ -0.7353,  -4.9236,  -5.1048,  ...,  -8.8734,  -8.8916,  -8.8550],
        [ -1.3056,  -5.3870,  -5.4945,  ...,  -9.4895,  -9.5039,  -9.4959],
        [ -2.7649,  -7.2201,  -9.0916,  ..., -11.3106, -11.3414, -11.2702],
        ...,
        [ -0.0768,  -4.8210,  -4.4374,  ...,  -8.0483,  -8.0502,  -7.9903],
        [ -2.7347,  -5.3650,  -7.2549,  ..., -10.0498, -10.0661,  -9.9886],
        [ -1.0991,  -4.2569,  -6.1267,  ...,  -8.6882,  -8.6889,  -8.627

In [41]:
start_logits = outputs.start_logits.cpu().numpy()
end_logits = outputs.end_logits.cpu().numpy()
start_logits, end_logits

(array([[ -2.2607286,  -5.1783237,  -5.2708955, ...,  -9.524334 ,
          -9.518299 ,  -9.528752 ],
        [ -2.5960705,  -5.548207 ,  -5.5313296, ...,  -9.959749 ,
          -9.953274 ,  -9.986032 ],
        [ -3.7127383,  -7.184838 ,  -8.538834 , ..., -11.655704 ,
         -11.657138 , -11.650537 ],
        ...,
        [ -2.0260274,  -4.416659 ,  -4.497997 , ...,  -8.147893 ,
          -8.153036 ,  -8.17596  ],
        [ -4.155299 ,  -5.8304253,  -7.1642623, ..., -10.525525 ,
         -10.525096 , -10.489034 ],
        [ -3.2000244,  -5.8162093,  -6.7249393, ...,  -9.493477 ,
          -9.503815 ,  -9.487099 ]], shape=(100, 384), dtype=float32),
 array([[ -0.73526084,  -4.9235563 ,  -5.1047883 , ...,  -8.873429  ,
          -8.891551  ,  -8.855039  ],
        [ -1.3056461 ,  -5.387043  ,  -5.494544  , ...,  -9.489532  ,
          -9.503882  ,  -9.495851  ],
        [ -2.7649236 ,  -7.220073  ,  -9.091585  , ..., -11.3106365 ,
         -11.34136   , -11.27022   ],
        ...,
   

In [46]:
import collections

example_to_features = collections.defaultdict(list)
for idx, feature in enumerate(eval_set):
    example_to_features[feature["example_id"]].append(idx)
eval_set, example_to_features

(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id'],
     num_rows: 100
 }),
 defaultdict(list,
             {'56be4db0acb8001400a502ec': [0],
              '56be4db0acb8001400a502ed': [1],
              '56be4db0acb8001400a502ee': [2],
              '56be4db0acb8001400a502ef': [3],
              '56be4db0acb8001400a502f0': [4],
              '56be8e613aeaaa14008c90d1': [5],
              '56be8e613aeaaa14008c90d2': [6],
              '56be8e613aeaaa14008c90d3': [7],
              '56bea9923aeaaa14008c91b9': [8],
              '56bea9923aeaaa14008c91ba': [9],
              '56bea9923aeaaa14008c91bb': [10],
              '56beace93aeaaa14008c91df': [11],
              '56beace93aeaaa14008c91e0': [12],
              '56beace93aeaaa14008c91e1': [13],
              '56beace93aeaaa14008c91e2': [14],
              '56beace93aeaaa14008c91e3': [15],
              '56bf10f43aeaaa14008c94fd': [16],
              '56bf10f43aeaaa14008c94fe': 

In [ ]:
import numpy as np

n_best = 20
max_answer_length = 30
predicted_answers = []

for example in small_eval_set:
    example_id = example["id"]
    context = example["context"]
    answers = []

    for feature_index in example_to_features[example_id]:
        start_logit = start_logits[feature_index]
        end_logit = end_logits[feature_index]
        offsets = eval_set["offset_mapping"][feature_index]

        start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
        end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
        for start_index in start_indexes:
            for end_index in end_indexes:
                # Skip answers that are not fully in the context
                if offsets[start_index] is None or offsets[end_index] is None:
                    continue
                # Skip answers with a length that is either < 0 or > max_answer_length.
                if (
                    end_index < start_index
                    or end_index - start_index + 1 > max_answer_length
                ):
                    continue

                answers.append(
                    {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                )

    best_answer = max(answers, key=lambda x: x["logit_score"])
    predicted_answers.append({"id": example_id, "prediction_text": best_answer["text"]})

In [ ]:
import evaluate

metric = evaluate.load("squad")

In [ ]:
theoretical_answers = [
    {"id": ex["id"], "answers": ex["answers"]} for ex in small_eval_set
]

In [ ]:
print(predicted_answers[0])
print(theoretical_answers[0])

{'id': '56be4db0acb8001400a502ec', 'prediction_text': 'Denver Broncos'}
{'id': '56be4db0acb8001400a502ec', 'answers': {'text': ['Denver Broncos', 'Denver Broncos', 'Denver Broncos'], 'answer_start': [177, 177, 177]}}

In [ ]:
metric.compute(predictions=predicted_answers, references=theoretical_answers)

{'exact_match': 83.0, 'f1': 88.25}

In [ ]:
from tqdm.auto import tqdm


def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        # Loop through all features associated with that example
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip answers that are not fully in the context
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # Skip answers with a length that is either < 0 or > max_answer_length
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)

        # Select the answer with the best score
        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            predicted_answers.append(
                {"id": example_id, "prediction_text": best_answer["text"]}
            )
        else:
            predicted_answers.append({"id": example_id, "prediction_text": ""})

    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [ ]:
compute_metrics(start_logits, end_logits, eval_set, small_eval_set)

{'exact_match': 83.0, 'f1': 88.25}

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
    "bert-finetuned-squad",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    push_to_hub=True,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    processing_class=tokenizer,
)
trainer.train()

In [ ]:
predictions, _ = trainer.predict(validation_dataset)
start_logits, end_logits = predictions
compute_metrics(start_logits, end_logits, validation_dataset, raw_datasets["validation"])

{'exact_match': 81.18259224219489, 'f1': 88.67381321905516}

In [ ]:
trainer.push_to_hub(commit_message="Training complete")

'https://huggingface.co/sgugger/bert-finetuned-squad/commit/9dcee1fbc25946a6ed4bb32efb1bd71d5fa90b68'

In [ ]:
from torch.utils.data import DataLoader
from transformers import default_data_collator

train_dataset.set_format("torch")
validation_set = validation_dataset.remove_columns(["example_id", "offset_mapping"])
validation_set.set_format("torch")

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    collate_fn=default_data_collator,
    batch_size=8,
)
eval_dataloader = DataLoader(
    validation_set, collate_fn=default_data_collator, batch_size=8
)

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
from accelerate import Accelerator

accelerator = Accelerator(fp16=True)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

In [ ]:
from transformers import get_scheduler

num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch

lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

In [ ]:
from huggingface_hub import HfApi, get_full_repo_name

model_name = "bert-finetuned-squad-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'sgugger/bert-finetuned-squad-accelerate'

In [ ]:
output_dir = "bert-finetuned-squad-accelerate"
api = HfApi()
api.create_repo(repo_name, exist_ok=True)

In [ ]:
from tqdm.auto import tqdm
import torch

progress_bar = tqdm(range(num_training_steps))

for epoch in range(num_train_epochs):
    # Training
    model.train()
    for step, batch in enumerate(train_dataloader):
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

    # Evaluation
    model.eval()
    start_logits = []
    end_logits = []
    accelerator.print("Evaluation!")
    for batch in tqdm(eval_dataloader):
        with torch.no_grad():
            outputs = model(**batch)

        start_logits.append(accelerator.gather(outputs.start_logits).cpu().numpy())
        end_logits.append(accelerator.gather(outputs.end_logits).cpu().numpy())

    start_logits = np.concatenate(start_logits)
    end_logits = np.concatenate(end_logits)
    start_logits = start_logits[: len(validation_dataset)]
    end_logits = end_logits[: len(validation_dataset)]

    metrics = compute_metrics(
        start_logits, end_logits, validation_dataset, raw_datasets["validation"]
    )
    print(f"epoch {epoch}:", metrics)

    # Save and upload
    accelerator.wait_for_everyone()
    unwrapped_model = accelerator.unwrap_model(model)
    unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)
    if accelerator.is_main_process:
        tokenizer.save_pretrained(output_dir)
        api.upload_folder(
            repo_id=repo_name,
            folder_path=output_dir,
            commit_message=f"Training in progress epoch {epoch}",
        )

In [ ]:
accelerator.wait_for_everyone()
unwrapped_model = accelerator.unwrap_model(model)
unwrapped_model.save_pretrained(output_dir, save_function=accelerator.save)

In [ ]:
from transformers import pipeline

# Replace this with your own checkpoint
model_checkpoint = "huggingface-course/bert-finetuned-squad"
question_answerer = pipeline("question-answering", model=model_checkpoint)

context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question = "Which deep learning libraries back 🤗 Transformers?"
question_answerer(question=question, context=context)

{'score': 0.9979003071784973,
 'start': 78,
 'end': 105,
 'answer': 'Jax, PyTorch and TensorFlow'}